# Camada Silver — Transformação e Limpeza

## Objetivo
Transformar as 5 tabelas Bronze em 7 tabelas Silver limpas, tipadas e em português,
prontas para modelagem dimensional na camada Gold.

In [0]:
catalog = "movies_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

catalog: movies_medallion
bronze_schema: movies_medallion.bronze
silver_schema: movies_medallion.silver
gold_schema: movies_medallion.gold
landing_path: /Volumes/movies_medallion/bronze/Landing


## silver.tb_info_filmes
Metadados principais dos filmes.
- Normalização e tradução de status
- Dedup por versão mais recente (ingestion_datetime)
- Conversão de data multi-formato com coalesce
- Derivada: ano_lancamento

In [0]:
from pyspark.sql import functions as F, Row
from pyspark.sql import Window
from datetime import datetime

# O dicionário para facilitar a tradução das colunas e eventual manutencão.
col_names_traslated = {
    "id": "id_filme",
    "title": "titulo",
    "original_title": "titulo_original",
    "release_date": "data_lancamento",
    "runtime": "duracao_minutos",
    "original_language": "idioma_original",
    "status": "status_filme",
    "overview": "sinopse",
    "tagline": "frase_divulgacao",
    "ingestion_datetime": "ingestao_datetime",
}

# Window para pegar o resultado mais recentemente ingerido de id_filme
w = Window.partitionBy("id_filme").orderBy(
    F.col("ingestao_datetime").desc(), F.col("id_filme").asc()
)

df_bronze_movies_info = spark.table(f"{bronze_schema}.tb_movies_info")

# Investiga e retorna quais os formatos possiveis da coluna release_date para posterior padronização
# Troca dígitos por "9", letras por "A" e agrupa
display(
    df_bronze_movies_info.select("release_date")
    .withColumn(
        "assinatura",
        F.regexp_replace(
            F.regexp_replace(F.trim(F.lower(F.col("release_date"))), r"\d", "9"),
            r"[a-z]",
            "A",
        ),
    )
    .groupBy("assinatura")
    .count()
    .orderBy(F.col("count").desc())
)

# Padronizações e transformações para criação da tabela silver
df_silver_movies_info = (
    # Renomeação com base no dicionário
    df_bronze_movies_info.select(list(col_names_traslated.keys()))
    .withColumnsRenamed(col_names_traslated)
    # Normalização da coluna status_filme e tradução de seus status
    .withColumn(
        "status_filme",
        F.trim(F.lower(F.regexp_replace("status_filme", r"[-\s]+", " "))),
    )
    .withColumn(
        "status_filme",
        F.when(F.col("status_filme").isNull(), "Não Informado")
        .when(F.col("status_filme") == "released", "Lançado")
        .when(F.col("status_filme") == "post production", "Pós-Produção")
        .when(F.col("status_filme") == "in production", "Em Produção")
        .when(F.col("status_filme") == "planned", "Planejado")
        .when(F.col("status_filme") == "rumored", "Rumores")
        .when(F.col("status_filme") == "canceled", "Cancelado")
        .otherwise("Não Informado")
    )
    .withColumn("duracao_minutos", F.col("duracao_minutos").try_cast("int"))
    # Usa a window criada para selecionar apenas o registro mais recente por id_filme (dedupação)
    .withColumn("rk", F.row_number().over(w))
    .filter(F.col("rk") == 1)
    .drop("rk")
    # Testa e padroniza os 3 formatos de datas identificados na inspeção
    .withColumn(
        "data_lancamento",
        F.coalesce(
            F.try_to_date(F.col("data_lancamento"), "yyyy-MM-dd"),
            F.try_to_date(F.col("data_lancamento"), "dd/MM/yyyy"),
            F.try_to_date(F.col("data_lancamento"), "dd-MM-yyyy"),
        ),
    )
    .withColumn("ano_lancamento", F.year(F.col("data_lancamento")))
    .drop("ingestao_datetime")
)

# Validaççoes: bronze vs silver, total vs únicos, nulls em data.
print("Bronze total:               ", df_bronze_movies_info.count())
print(
    "Bronze ids únicos:          ",
    df_bronze_movies_info.select("id").distinct().count(),
)
print("Silver total:               ", df_silver_movies_info.count())
print(
    "Silver ids únicos:          ",
    df_silver_movies_info.select("id_filme").distinct().count(),
)

display(
    df_silver_movies_info.select(
        F.count("*").alias("total"),
        F.sum(F.when(F.col("data_lancamento").isNull(), 1).otherwise(0)).alias("nulls"),
    )
)

display(
    df_silver_movies_info.groupBy("data_lancamento")
    .count()
    .orderBy("count", ascending=False)
)

print("Total de colunas:", len(df_silver_movies_info.columns))
df_silver_movies_info.printSchema()

# Persiste o DataFrame como tabela Delta no schema silver.
(df_silver_movies_info
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_info_filmes"))

print(f"Tabela gravada: {silver_schema}.tb_info_filmes")

# Validção da tabela gravada
df_check = spark.table(f"{silver_schema}.tb_info_filmes")
print(f"Linhas na tabela: {df_check.count()}")
df_check.printSchema()
display(df_check.limit(5))

assinatura,count
9999-99-99,91541
99/99/9999,10602
99-99-9999,4717
null,62
AAAAA'A AAAAAAAA AAAAAAAAA. AAAAA'A AAAAAAAA AAAAAAA.,1
AAA AAAAAA AAAA AAAAAAAAA AAAAAAA,1
AAAAA AAAAAAAA AA AAAAAAAA AAA AAAA AA AAA AAAAAAA,1
AA AAAAA AAA A AAAAA AAAAAA,1
AA AAAAAAAAAAA AAAA AAAAA AAAAA AAAAA AAAAAAA AAAAAAAAAA AAAAAAAAA,1
AAA AAAA AAAAAA AAAAAAAAA AAA AAAAAAA AAAA,1


Bronze total:                106930
Bronze ids únicos:           97879
Silver total:                97879
Silver ids únicos:           97879


total,nulls
97879,2565


data_lancamento,count
null,2565
2016-01-01,469
2017-01-01,374
2018-01-01,284
2019-01-01,177
2020-01-01,173
2021-01-01,135
2020-10-01,128
2019-11-01,124
2019-10-18,118


Total de colunas: 10
root
 |-- id_filme: integer (nullable = true)
 |-- titulo: string (nullable = true)
 |-- titulo_original: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = false)
 |-- sinopse: string (nullable = true)
 |-- frase_divulgacao: string (nullable = true)
 |-- ano_lancamento: integer (nullable = true)

Tabela gravada: movies_medallion.silver.tb_info_filmes
Linhas na tabela: 97879
root
 |-- id_filme: integer (nullable = true)
 |-- titulo: string (nullable = true)
 |-- titulo_original: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = true)
 |-- sinopse: string (nullable = true)
 |-- frase_divulgacao: string (nullable = true)
 |-- ano_lancamento: integer (nullable = true)


id_filme,titulo,titulo_original,data_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao,ano_lancamento
14564,Rings,Rings,2017-02-01,102,en,Lançado,"\Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a """"\""""movie within the movie""""\"""" that no one has ever seen before.""""""""First you watch it. Then you die.",null,2017
32471,Mixtape,Mixtape,2021-12-03,94,en,Lançado,null,null,2021
38258,Grizzly II: Revenge,Grizzly II: Revenge,2020-02-17,74,en,Lançado,\All hell breaks loose when a giant grizzly,reacting to the slaughter of her cubs by poachers,2020
38492,Billy Joel - Live at Yankee Stadium,Billy Joel Live at Yankee Stadium,2022-06-22,86,en,Lançado,Billy Joel plays his greatest hits in the Big Apple.,null,2022
38700,Bad Boys for Life,Bad Boys for Life,2020-01-15,124,en,Lançado,"Marcus and Mike are forced to confront new threats, career changes, and midlife crises as they join the newly created elite team AMMO of the Miami police department to take down the ruthless Armando Armas, the vicious leader of a Miami drug cartel.",Ride together. Die together.,2020


## silver.tb_financeiro_filmes
Métricas financeiras dos filmes em USD e BRL.
- Conversão segura de texto → decimal
- Conversão USD → BRL pela cotação mais recente
- Derivadas: lucro_usd, lucro_brl, margem_lucro_percentual

In [0]:
# Limpa e converte valores monetários.
def normalize_monetary_value(column_name):
    col = F.col(column_name)
    cleaned = F.regexp_replace(col, r"[^\d.-]", "")
    converted = cleaned.try_cast("decimal(18,2)")
    return F.when(converted > 0, converted)


df_bronze_movies_financials = spark.table(f"{bronze_schema}.tb_movies_financials")

# O dicionário para facilitar a tradução das colunas e eventual manutencão.
col_names_translated = {
    "id": "id_filme",
    "budget": "orcamento_usd",
    "revenue": "receita_usd",
}

# Padronizações e transformações para criação da tabela silver
df_silver_movies_financials = (
    df_bronze_movies_financials.select(list(col_names_translated.keys()))
    .withColumnsRenamed(col_names_translated)
    .withColumn("orcamento_usd", normalize_monetary_value("orcamento_usd"))
    .withColumn("receita_usd", normalize_monetary_value("receita_usd"))
)

# Validações e checagens
display(
    df_silver_movies_financials.select(
        F.count("*").alias("total"),
        F.sum(F.when(F.col("orcamento_usd").isNull(), 1).otherwise(0)).alias(
            "orc_nulos"
        ),
        F.sum(F.when(F.col("receita_usd").isNull(), 1).otherwise(0)).alias("rec_nulos"),
        F.max("orcamento_usd").alias("orc_max"),
        F.max("receita_usd").alias("rec_max"),
    )
)

display(
    df_silver_movies_financials.orderBy(F.col("receita_usd").desc_nulls_last())
    .select("id_filme", "orcamento_usd", "receita_usd")
    .limit(10)
)

df_bronze_cotacao_dolar = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

# Pegar cotação mais atual da tabela bronze
latest_rate = (
    df_bronze_cotacao_dolar.orderBy(F.col("dataHoraCotacao").desc())
    .select("cotacaoCompra")
    .first()["cotacaoCompra"]
)

# Calcula valores monetários em BRL e o percetual de lucro
df_silver_movies_financials = (
    df_silver_movies_financials.withColumn(
        "orcamento_brl", F.col("orcamento_usd") * latest_rate
    )
    .withColumn("receita_brl", F.col("receita_usd") * latest_rate)
    .withColumn("lucro_usd", F.col("receita_usd") - F.col("orcamento_usd"))
    .withColumn("lucro_brl", F.col("receita_brl") - F.col("orcamento_brl"))
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            F.col("receita_usd").isNotNull() & (F.col("receita_usd") > 0),
            F.round((F.col("lucro_usd") / F.col("receita_usd")) * 100, 2),
        ),
    )
)

display(
    df_silver_movies_financials.select(
        F.count("*").alias("total"),
        F.sum(F.when(F.col("orcamento_brl").isNull(), 1).otherwise(0)).alias(
            "orc_nulos"
        ),
        F.sum(F.when(F.col("receita_brl").isNull(), 1).otherwise(0)).alias("rec_nulos"),
        F.max("orcamento_brl").alias("orc_max"),
        F.max("receita_brl").alias("rec_max"),
    )
)

display(
    df_silver_movies_financials.orderBy(F.col("receita_brl").desc_nulls_last()).limit(
        10
    )
)

assert len(df_silver_movies_financials.columns) == 8, "Número de colunas inesperado"
print("Financeiro OK — 8 colunas")

# Persiste a tabela silver no catálogo
(df_silver_movies_financials
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_financeiro_filmes"))

print(f"Tabela gravada: {silver_schema}.tb_financeiro_filmes")

df_check = spark.table(f"{silver_schema}.tb_financeiro_filmes")
print(f"Linhas na tabela: {df_check.count()}")
df_check.printSchema()

total,orc_nulos,rec_nulos,orc_max,rec_max
106165,97281,102583,888000000.00,2800000000.00


id_filme,orcamento_usd,receita_usd
299534,356000000.00,2800000000.00
76600,460000000.00,2320250281.00
299536,300000000.00,2052415039.00
634649,200000000.00,1921847111.00
420818,260000000.00,1663075401.00
361743,170000000.00,1488732821.00
346698,145.00,1428545028.00
502356,100000000.00,1355725263.00
284054,200000000.00,1349926083.00
181808,200000000.00,1332698830.00


total,orc_nulos,rec_nulos,orc_max,rec_max
106165,97281,102583,4.4126496E9,1.391376E10


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
299534,356000000.00,2800000000.00,1.7690352E9,1.391376E10,2444000000.00,1.21447248E10,87.29
76600,460000000.00,2320250281.00,2.285832E9,1.15297876963452E10,1860250281.00,9.2439556963452E9,80.17
299536,300000000.00,2052415039.00,1.49076E9,1.01988608117988E10,1752415039.00,8.7081008117988E9,85.38
634649,200000000.00,1921847111.00,9.9384E8,9.5500426639812E9,1721847111.00,8.556202663981199E9,89.59
420818,260000000.00,1663075401.00,1.291992E9,8.2641542826491995E9,1403075401.00,6.9721622826491995E9,84.37
361743,170000000.00,1488732821.00,8.44764E8,7.3978111341132E9,1318732821.00,6.5530471341132E9,88.58
346698,145.00,1428545028.00,720.534,7.0987259531376E9,1428544883.00,7.0987252326036E9,100.00
502356,100000000.00,1355725263.00,4.9692E8,6.7368699768996E9,1255725263.00,6.2399499768996E9,92.62
284054,200000000.00,1349926083.00,9.9384E8,6.7080526916435995E9,1149926083.00,5.7142126916435995E9,85.18
181808,200000000.00,1332698830.00,9.9384E8,6.622447026036E9,1132698830.00,5.628607026036E9,84.99


Financeiro OK — 8 colunas
Tabela gravada: movies_medallion.silver.tb_financeiro_filmes
Linhas na tabela: 106165
root
 |-- id_filme: integer (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: double (nullable = true)
 |-- receita_brl: double (nullable = true)
 |-- lucro_usd: decimal(19,2) (nullable = true)
 |-- lucro_brl: double (nullable = true)
 |-- margem_lucro_percentual: decimal(26,2) (nullable = true)



## silver.tb_metricas_engajamento

Métricas de popularidade, notas e votos (TMDB e IMDb).

- Limpeza de separadores decimais mistos (`,` e `.`)
- Validação de faixa: notas 0-10, votos e popularidade ≥ 0
- Conversão segura via `try_cast` (inválidos → NULL)

In [0]:
# Limpam, processam e convertem para o formato necessário
def normalize_rating_decimal(column_name):
    col = F.col(column_name)
    cleaned = F.regexp_replace(col, r"[^\d.,-]", "")
    normalized = F.regexp_replace(cleaned, r",", ".")
    single_dot = F.regexp_replace(normalized, r"\.(?=.*\.)", "")
    converted = single_dot.try_cast("decimal(18,2)")
    return F.when((converted >= 0) & (converted <= 10), converted)


def normalize_positive_integer(column_name):
    col = F.col(column_name)
    cleaned = F.regexp_replace(col, r"[^\d.,-]", "")
    normalized = F.regexp_replace(cleaned, r",", ".")
    single_dot = F.regexp_replace(normalized, r"\.(?=.*\.)", "")
    converted = single_dot.try_cast("integer")
    return F.when(converted >= 0, converted)


def normalize_positive_decimal(column_name):
    col = F.col(column_name)
    cleaned = F.regexp_replace(col, r"[^\d.,-]", "")
    normalized = F.regexp_replace(cleaned, r",", ".")
    single_dot = F.regexp_replace(normalized, r"\.(?=.*\.)", "")
    converted = single_dot.try_cast("decimal(18,2)")
    return F.when(converted >= 0, converted)

df_bronze_movies_metrics = spark.table(f"{bronze_schema}.tb_movies_metrics")

display(df_bronze_movies_metrics.limit(50))

# O dicionário para facilitar a tradução das colunas e eventual manutencão.
col_names_translated = {
    "id": "id_filme",
    "popularity": "popularidade",
    "vote_average": "nota_media_tmdb",
    "vote_count": "qtd_votos_tmdb",
    "averageRating": "nota_media_imdb",
    "numVotes": "qtd_votos_imdb",
}

# Padronizações e transformações para criação da tabela silver
df_silver_movies_metrics = (
    df_bronze_movies_metrics.select(list(col_names_translated.keys()))
    .withColumnsRenamed(col_names_translated)
    .withColumn("popularidade", normalize_positive_decimal("popularidade"))
    .withColumn("nota_media_tmdb", normalize_rating_decimal("nota_media_tmdb"))
    .withColumn("qtd_votos_tmdb", normalize_positive_integer("qtd_votos_tmdb"))
    .withColumn("nota_media_imdb", normalize_rating_decimal("nota_media_imdb"))
    .withColumn("qtd_votos_imdb", normalize_positive_integer("qtd_votos_imdb"))
)

display(df_silver_movies_metrics.limit(50))

# Persiste no catálogo silver.
(df_silver_movies_metrics
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_metricas_engajamento"))

print(f"Tabela gravada: {silver_schema}.tb_metricas_engajamento")

df_check = spark.table(f"{silver_schema}.tb_metricas_engajamento")
print(f"Linhas na tabela: {df_check.count()}")
df_check.printSchema()
display(df_check.limit(5))



id,popularity,vote_average,vote_count,averageRating,numVotes,ingestion_datetime
293660,72.735,7.606,28894,8.0,1270339,2026-10-07T20:20:33.039Z
299536,"154,34",8.255,27713,8.4,1406782,2026-10-07T20:20:33.039Z
299534,91.756,8.263,23857,8.4,1484150,2026-10-07T20:20:33.039Z
475557,"54,522",8.168,23425,8.3,1723035,2026-10-07T20:20:33.039Z
271110,70.741,7.4,21541,7.8,947222,2026-10-07T20:20:33.039Z
284054,43.665,7.39,null,7.3,924922,2026-10-07T20:20:33.039Z
284052,70.535,7.427,20935,7.5,895880,2026-10-07T20:20:33.039Z
315635,65.88,7.345,20507,7.4,835116,2026-10-07T20:20:33.039Z
283995,67.553,7.624,20353,7.6,844767,2026-10-07T20:20:33.039Z
297761,35.356,5.909,20097,5.9,null,2026-10-07T20:20:33.039Z


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
293660,72.74,7.61,28894,8.00,1270339
299536,154.34,8.26,27713,8.40,1406782
299534,91.76,8.26,23857,8.40,1484150
475557,54.52,8.17,23425,8.30,1723035
271110,70.74,7.40,21541,7.80,947222
284054,43.67,7.39,null,7.30,924922
284052,70.54,7.43,20935,7.50,895880
315635,65.88,7.35,20507,7.40,835116
283995,67.55,7.62,20353,7.60,844767
297761,35.36,5.91,20097,5.90,null


Tabela gravada: movies_medallion.silver.tb_metricas_engajamento
Linhas na tabela: 107364
root
 |-- id_filme: integer (nullable = true)
 |-- popularidade: decimal(18,2) (nullable = true)
 |-- nota_media_tmdb: decimal(18,2) (nullable = true)
 |-- qtd_votos_tmdb: integer (nullable = true)
 |-- nota_media_imdb: decimal(18,2) (nullable = true)
 |-- qtd_votos_imdb: integer (nullable = true)



id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
293660,72.74,7.61,28894,8.00,1270339
299536,154.34,8.26,27713,8.40,1406782
299534,91.76,8.26,23857,8.40,1484150
475557,54.52,8.17,23425,8.30,1723035
271110,70.74,7.40,21541,7.80,947222


## silver.tb_avaliacoes_usuarios

Avaliações individuais de usuários por filme.

- Validação da nota: 0-10
- Comentários vazios → "Sem comentário"
- Deduplicação por filme + usuário + nota + comentário

In [0]:
df_bronze_movies_reviews = spark.table(f"{bronze_schema}.tb_movies_reviews")

df_bronze_movies_reviews.printSchema()
display(df_bronze_movies_reviews.limit(50))

# O dicionário para facilitar a tradução das colunas e eventual manutencão.
col_names_translated = {
    "id": "id_filme",
    "nome": "nome_usuario",
    "nota": "nota_usuario",
    "comentario": "comentario_usuario",
}

# Padronizações e transformações para criação da tabela silver
df_silver_movies_reviews = (
    df_bronze_movies_reviews
    .select(list(col_names_translated.keys()))
    .withColumnsRenamed(col_names_translated)
    .withColumn(
        "nota_usuario",
        F.when(
            (F.col("nota_usuario") >= 0) & (F.col("nota_usuario") <= 10),
            F.col("nota_usuario"),
        ),
    )
    .withColumn("comentario_usuario", F.trim(F.col("comentario_usuario")))
    .withColumn(
        "comentario_usuario",
        F.when(F.length(F.col("comentario_usuario")) > 0, F.col("comentario_usuario"))
        .otherwise("Sem comentário"),
    )
    .dropDuplicates(
        ["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"]
    )
)

display(df_silver_movies_reviews.limit(20))

(df_silver_movies_reviews
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios"))

print(f"Tabela gravada: {silver_schema}.tb_avaliacoes_usuarios")

df_check = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")
print(f"Linhas na tabela: {df_check.count()}")
df_check.printSchema()

root
 |-- id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- nota: double (nullable = true)
 |-- comentario: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



id,nome,nota,comentario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,null,2026-10-07T20:20:42.187Z
637007,Lucas Reis 602,3.9,null,2026-10-07T20:20:42.187Z
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.,2026-10-07T20:20:42.187Z
413036,Gabriela Monteiro 401,7.5,null,2026-10-07T20:20:42.187Z
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.,2026-10-07T20:20:42.187Z
387727,Maria Alves 707,8.2,"Gostei bastante, recomendo.",2026-10-07T20:20:42.187Z
1032506,Amanda Castro 242,4.7,"Fraco, não recomendo.",2026-10-07T20:20:42.187Z
446554,Sandra Rodrigues 736,6.5,Assisti até o final mas não me marcou.,2026-10-07T20:20:42.187Z
569916,Camila Lima 489,8.2,Muito bom! Vale a pena assistir.,2026-10-07T20:20:42.187Z
864552,Roberto Almeida 555,9.3,"Obra-prima do cinema, simplesmente espetacular.",2026-10-07T20:20:42.187Z


id_filme,nome_usuario,nota_usuario,comentario_usuario
442113,Mariana Cardoso 277,4.4,Sem comentário
637007,Lucas Reis 602,3.9,Sem comentário
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.
413036,Gabriela Monteiro 401,7.5,Sem comentário
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.
387727,Maria Alves 707,8.2,"Gostei bastante, recomendo."
1032506,Amanda Castro 242,4.7,"Fraco, não recomendo."
446554,Sandra Rodrigues 736,6.5,Assisti até o final mas não me marcou.
569916,Camila Lima 489,8.2,Muito bom! Vale a pena assistir.
864552,Roberto Almeida 555,9.3,"Obra-prima do cinema, simplesmente espetacular."


Tabela gravada: movies_medallion.silver.tb_avaliacoes_usuarios
Linhas na tabela: 32412
root
 |-- id_filme: integer (nullable = true)
 |-- nome_usuario: string (nullable = true)
 |-- nota_usuario: double (nullable = true)
 |-- comentario_usuario: string (nullable = true)



## silver.tb_generos
Catálogo de gêneros por filme (relação filme ↔ gênero).

- Padronização de separadores via regex
- Explode em uma linha por gênero
- Whitelist com os 19 gêneros válidos do TMDB

In [0]:
df_bronze_credits_and_tags = spark.table(f"{bronze_schema}.tb_credits_and_tags")

df_bronze_credits_and_tags.printSchema()
display(df_bronze_credits_and_tags.limit(50))


# O dicionário para facilitar a tradução das colunas e eventual manutencão.
col_names_translated = {
    "id": "id_filme",
    "genres": "generos",
}

# Lista dos 19 gêneros oficiais do TMDB.
VALID_GENRES = [
    "Action",
    "Adventure",
    "Animation",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Family",
    "Fantasy",
    "History",
    "Horror",
    "Music",
    "Mystery",
    "Romance",
    "Science Fiction",
    "TV Movie",
    "Thriller",
    "War",
    "Western",
]

# Padronizações e transformações para criação da tabela silver
df_silver_generos = (
    df_bronze_credits_and_tags.select(list(col_names_translated.keys()))
    .withColumnsRenamed(col_names_translated)
    .select("id_filme", "generos")
    .withColumn("generos", F.regexp_replace(F.col("generos"), r"[^a-zA-ZÀ-ÿ, ]", ","))
    .withColumn("generos", F.explode(F.split(F.col("generos"), ",")))
    .withColumn("generos", F.trim(F.col("generos")))
    .filter(F.length(F.col("generos")) > 1)
    .filter(F.col("generos").isin(*VALID_GENRES))
)

display(
    df_silver_generos.groupBy("generos")
    .count()
    .orderBy(F.col("count").desc())
    .limit(30)
)

print("Filmes únicos no bronze:", df_bronze_credits_and_tags.select("id").distinct().count())
print("Filmes únicos no silver:", df_silver_generos.select("id_filme").distinct().count())

(df_silver_generos
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_generos"))

print(f"Tabela gravada: {silver_schema}.tb_generos")

df_check = spark.table(f"{silver_schema}.tb_generos")
print(f"Linhas na tabela: {df_check.count()}")
df_check.printSchema()


root
 |-- id: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_companies: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- spoken_languages: string (nullable = true)
 |-- keywords: string (nullable = true)
 |-- directors: string (nullable = true)
 |-- writers: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



id,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,cast,ingestion_datetime
293660,"Action, Adventure, Comedy","20th Century Fox, The Donners' Company, Genre Films",United States of America,English,"superhero, anti hero, mercenary, based on comic, aftercreditsstinger, duringcreditsstinger",Tim Miller,"Rhett Reese, Paul Wernick","Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder",2026-10-07T20:20:37.210Z
299536,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, space, battlefield, genocide, magical object, super power, aftercreditsstinger, marvel cinematic universe (mcu), cosmic","Anthony Russo, Joe Russo",N/A,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman",2026-10-07T20:20:37.210Z
299534,"Adventure, Science Fiction, Action",Marvel Studios,United States of America,"English, Japanese, Xhosa","superhero, time travel, space travel, time machine, based on comic, sequel, alien invasion, superhero team, marvel cinematic universe (mcu), alternate timeline, father daughter relationship, sister sister relationship","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Josh Brolin, Don Cheadle, Paul Rudd, Benedict Cumberbatch",2026-10-07T20:20:37.210Z
475557,"Crime, Thriller, Drama","Warner Bros. Pictures, Joint Effort, Village Roadshow Pictures, Bron Studios, DC Films","Canada, United States of America",English,"dream, street gang, society, psychopath, clown, villain, based on comic, murder, psychological thriller, criminal mastermind, mental illness, anarchy, character study, clown makeup, subway train, social realism, supervillain, tv host, 1980s, mother son relationship, origin story, falling into madness, depressing",Todd Phillips,"Todd Phillips, Scott Silver, Bob Kane, Bill Finger, Jerry Robinson","Joaquin Phoenix, Robert De Niro, Zazie Beetz, Frances Conroy, Brett Cullen, Shea Whigham, Bill Camp, Glenn Fleshler, Leigh Gill, Josh Pais",2026-10-07T20:20:37.210Z
271110,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"Romanian, English, German, Russian","civil war, superhero, based on comic, sequel, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu), excited","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Joe Simon, Jack Kirby","Chris Evans, Robert Downey Jr., Scarlett Johansson, Sebastian Stan, Anthony Mackie, Don Cheadle, Jeremy Renner, Chadwick Boseman, Paul Bettany, Elizabeth Olsen",2026-10-07T20:20:37.210Z
284054,"Action, Adventure, Science Fiction",Marvel Studios,United States of America,"English, Korean, Swahili, Xhosa","africa, superhero, based on comic, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu)",Ryan Coogler,"Ryan Coogler, Joe Robert Cole, Stan Lee, Jack Kirby","Chadwick Boseman, Michael B. Jordan, Lupita Nyong'o, Danai Gurira, Martin Freeman, Daniel Kaluuya, Letitia Wright, Winston Duke, Sterling K. Brown, Angela Bassett",2026-10-07T20:20:37.210Z
284052,"Action, Adventure, Fantasy",Marvel Studios,UNITED STATES OF AMERICA,English,"magic, superhero, training, time, based on comic, sorcerer, doctor, neurosurgeon, wizard, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu)",Scott Derrickson,N/A,"Benedict Cumberbatch, Chiwetel Ejiofor, Rachel McAdams, Benedict Wong, Mads Mikkelsen, Tilda Swinton, Michael Stuhlbarg, Benjamin Bratt, Scott Adkins, Zara Phythian",2026-10-07T20:20:37.210Z
315635,"Action, Adventure, Science Fiction, Drama","Ma

generos,count
Drama,34144
Comedy,21154
Documentary,20018
Thriller,11263
Horror,11198
Action,8005
Romance,7996
Animation,5439
Crime,5219
Science Fiction,4376


Filmes únicos no bronze: 99006
Filmes únicos no silver: 85155
Tabela gravada: movies_medallion.silver.tb_generos
Linhas na tabela: 154843
root
 |-- id_filme: integer (nullable = true)
 |-- generos: string (nullable = true)



## silver.tb_pessoas_empresas
Dimensão unificada de entidades categorizada por tipo.

- Explode de cada coluna em linhas individuais
- Padronização de capitalização (`initcap`) e `trim`
- Categorização: Ator, Diretor, Roteirista, Produtora
- União das 4 fontes e deduplicação

In [0]:
df_bronze_credits_and_tags = spark.table(f"{bronze_schema}.tb_credits_and_tags")

# Padronizações e transformações para criação das tabelas que serão unidas
df_cast = (
    df_bronze_credits_and_tags
    .select(
        F.col("id").alias("id_filme"),
        F.explode(F.split(F.col("cast"), ",")).alias("nome"),
    )
    .withColumn("nome", F.trim(F.col("nome")))
    .withColumn("nome", F.initcap(F.col("nome")))
    .withColumn("tipo", F.lit("Ator"))
    .filter(F.length(F.col("nome")) > 1)
)

df_directors = (
    df_bronze_credits_and_tags
    .select(
        F.col("id").alias("id_filme"),
        F.explode(F.split(F.col("directors"), ",")).alias("nome"),
    )
    .withColumn("nome", F.trim(F.col("nome")))
    .withColumn("nome", F.initcap(F.col("nome")))
    .withColumn("tipo", F.lit("Diretor"))
    .filter(F.length(F.col("nome")) > 1)
)

df_writers = (
    df_bronze_credits_and_tags
    .select(
        F.col("id").alias("id_filme"),
        F.explode(F.split(F.col("writers"), ",")).alias("nome"),
    )
    .withColumn("nome", F.trim(F.col("nome")))
    .withColumn("nome", F.initcap(F.col("nome")))
    .withColumn("tipo", F.lit("Roteirista"))
    .filter(F.length(F.col("nome")) > 1)
)

df_production_companies = (
    df_bronze_credits_and_tags
    .select(
        F.col("id").alias("id_filme"),
        F.explode(F.split(F.col("production_companies"), ",")).alias("nome"),
    )
    .withColumn("nome", F.trim(F.col("nome")))
    .withColumn("nome", F.initcap(F.col("nome")))
    .withColumn("tipo", F.lit("Produtora"))
    .filter(F.length(F.col("nome")) > 1)
)

# Criação da tabela de dominio final
df_silver_pessoas_empresas = (
    df_cast
    .unionByName(df_directors)
    .unionByName(df_writers)
    .unionByName(df_production_companies)
    .dropDuplicates(["id_filme", "nome", "tipo"])
)

display(df_silver_pessoas_empresas.limit(50))

display(
    df_silver_pessoas_empresas
    .groupBy("tipo")
    .count()
    .orderBy(F.col("count").desc())
)

df_silver_pessoas_empresas.printSchema()

(df_silver_pessoas_empresas
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_pessoas_empresas"))

print(f"Tabela gravada: {silver_schema}.tb_pessoas_empresas")

df_check = spark.table(f"{silver_schema}.tb_pessoas_empresas")
print(f"Linhas na tabela: {df_check.count()}")
df_check.printSchema()



id_filme,nome,tipo
299536,Chris Hemsworth,Ator
475557,Frances Conroy,Ator
634649,Tom Holland,Ator
209112,Jeremy Irons,Ator
329865,Julia Scarlett Dan,Ator
381288,Anya Taylor-joy,Ator
419430,Caleb Landry Jones,Ator
424694,Ben Hardy,Ator
335983,Malcolm C. Murray,Ator
429617,Samuel L. Jackson,Ator


tipo,count
Ator,549156
Roteirista,138778
Produtora,120641
Diretor,109214


root
 |-- id_filme: integer (nullable = true)
 |-- nome: string (nullable = false)
 |-- tipo: string (nullable = false)

Tabela gravada: movies_medallion.silver.tb_pessoas_empresas
Linhas na tabela: 917789
root
 |-- id_filme: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- tipo: string (nullable = true)



## silver.tb_cotacao_dolar

Série temporal contínua da cotação USD/BRL.

- Conversão de data para `date` (remove hora)
- Geração de calendário diário entre data mínima e máxima
- Forward fill via `last(ignorenulls=True)` para fins de semana e feriados


In [0]:
df_bronze_cotacao_dolar = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

col_names_translated = {
    "cotacaoCompra": "cotacao_compra",
    "dataHoraCotacao": "data_cotacao",
}

df_silver_cotacao_dolar = (
    df_bronze_cotacao_dolar
    .select(list(col_names_translated.keys()))
    .withColumnsRenamed(col_names_translated)
    .withColumn("data_cotacao", F.to_date(F.col("data_cotacao")))
)

date_range = df_silver_cotacao_dolar.agg(
    F.min("data_cotacao").alias("min_date"),
    F.max("data_cotacao").alias("max_date"),
).first()

df_calendar = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(date_range["min_date"]),
                F.lit(date_range["max_date"]),
                F.expr("interval 1 day"),
            )
        ).alias("data_cotacao")
    )
)

w = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_silver_cotacao_dolar = (
    df_calendar
    .join(df_silver_cotacao_dolar, on="data_cotacao", how="left")
    .withColumn(
        "cotacao_compra",
        F.last("cotacao_compra", ignorenulls=True).over(w),
    )
)

display(df_silver_cotacao_dolar.orderBy("data_cotacao"))

data_cotacao,cotacao_compra
2026-09-29,5.2198
2026-09-30,5.1803
2026-10-01,5.2073
2026-10-02,5.2232
2026-10-03,5.2232
2026-10-04,5.2232
2026-10-05,4.9853
2026-10-06,4.9692
